In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('Combined Data.csv')
print(df.head())

   Unnamed: 0                                          statement   status
0           0                                         oh my gosh  Anxiety
1           1  trouble sleeping, confused mind, restless hear...  Anxiety
2           2  All wrong, back off dear, forward doubt. Stay ...  Anxiety
3           3  I've shifted my focus to something else but I'...  Anxiety
4           4  I'm restless and restless, it's been a month n...  Anxiety


In [3]:
df.drop('Unnamed: 0', axis=1, inplace=True)

In [4]:
df.shape

(53043, 2)

In [5]:
df.isnull().sum()/df.shape[0]*100

statement    0.682465
status       0.000000
dtype: float64

In [6]:
df.duplicated(['statement']).sum()

np.int64(1969)

In [7]:
df.dropna(inplace=True)

In [8]:
df.shape

(52681, 2)

In [9]:
df.drop_duplicates(inplace=True)

In [10]:
df.shape

(51093, 2)

In [11]:
df['status'].value_counts()/df.shape[0]*100

status
Normal                  31.393733
Depression              29.542207
Suicidal                20.832599
Anxiety                  7.090991
Bipolar                  4.894995
Stress                   4.493766
Personality disorder     1.751708
Name: count, dtype: float64

In [12]:
df['status'].unique()

array(['Anxiety', 'Normal', 'Depression', 'Suicidal', 'Stress', 'Bipolar',
       'Personality disorder'], dtype=object)

In [13]:
df['status'].value_counts()

status
Normal                  16040
Depression              15094
Suicidal                10644
Anxiety                  3623
Bipolar                  2501
Stress                   2296
Personality disorder      895
Name: count, dtype: int64

In [14]:
# Pass a list of values you want to change to 'Stress'
df['status'] = df['status'].replace(['Anxiety', 'Bipolar', 'Personality disorder'], 'Stress')
df['status'].value_counts()/df.shape[0]*100

status
Normal        31.393733
Depression    29.542207
Suicidal      20.832599
Stress        18.231460
Name: count, dtype: float64

In [15]:
import string
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kumar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [16]:
import string

In [17]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()

In [18]:
def preprocess_text(text):
    text = text.lower()  
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # Remove numbers
    text = re.sub(r'\d+', '', text)


    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Remove extra whitespace
    text = ' '.join(text.split())

    #tokenization
    tokens = text.split()

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    tokens = ' '.join(tokens)

    # Stemming
    text = [ps.stem(word) for word in tokens.split()]
    text = ' '.join(text)

    return text

In [19]:
df['statement'] = df['statement'].apply(preprocess_text)
print(df.head())

                                           statement  status
0                                            oh gosh  Stress
1       troubl sleep confus mind restless heart tune  Stress
2  wrong back dear forward doubt stay restless re...  Stress
3           ive shift focu someth els im still worri  Stress
4                im restless restless month boy mean  Stress


In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
tfidf = TfidfVectorizer(max_features=150, ngram_range=(1,2))


x = df['statement']
y = df['status']

In [21]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [22]:
x_train = tfidf.fit_transform(x_train)
x_test = tfidf.transform(x_test)
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [23]:
import joblib

joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
joblib.dump(le, 'label_encoder.pkl')
joblib.dump(x_train, 'x_train.joblib')
joblib.dump(x_test, 'x_test.joblib')
joblib.dump(y_train, 'y_train.joblib')
joblib.dump(y_test, 'y_test.joblib')

['y_test.joblib']

In [24]:
text = "I need help with my anxiety and stress levels."
processed_text = preprocess_text(text)
processed_text_vectorized = tfidf.transform([processed_text])
print(processed_text_vectorized)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4 stored elements and shape (1, 150)>
  Coords	Values
  (0, 6)	0.491214578023205
  (0, 54)	0.42089174832560167
  (0, 86)	0.4544316629880537
  (0, 120)	0.6124134536428922
